# Exp 5 — Signal Alignment: NLI + UMLS Density + UMLS Specificity

This notebook combines the three independent signals produced by the pipeline and visualises how they align across a reasoning chain.

| Stage | What it does |
|---|---|
| 0 | Environment setup & imports |
| 1 | CoT Generator — live LLM call |
| 2 | Concept Extractor — UMLS concept linking |
| 3 | Hybrid NLI Entailment Checker |
| 4 | Guard Signal Derivation |
| 5 | UMLS Signal Computation (density + specificity) |
| 6 | Signal Alignment & Visualisation |

Run cells top-to-bottom. Stages 0–4 are identical to `test_pipeline_notebook.ipynb`.  
Stages 5–6 are new.

## Stage 0 — Environment Setup

Clones the repo, installs dependencies, configures API keys. **Run this first.**

In [ ]:
# ── 0a. Clone / pull the repo and configure paths ─────────────────────────
import os, sys
from pathlib import Path

REPO_URL  = 'https://github.com/varchanaiyer/biomedical-semantic-leakage-detection'
REPO_DIR  = 'biomedical-semantic-leakage-detection'

if not Path(REPO_DIR).exists():
    os.system(f'git clone {REPO_URL}')
else:
    os.system(f'git -C {REPO_DIR} pull --quiet')

_cwd = Path(os.getcwd())
if (_cwd / REPO_DIR / 'utils').exists():
    PROJECT_ROOT = str(_cwd / REPO_DIR)
elif (_cwd / 'utils').exists():
    PROJECT_ROOT = str(_cwd)
elif (_cwd.parent / 'utils').exists():
    PROJECT_ROOT = str(_cwd.parent)
else:
    PROJECT_ROOT = str(_cwd / REPO_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")
print(f"Working dir  : {os.getcwd()}")

In [ ]:
# ── 0b. Install dependencies ───────────────────────────────────────────────
!pip install openai numpy pandas scipy scikit-learn matplotlib seaborn requests ipywidgets --quiet
print("Dependencies installed.")

In [ ]:
# ── 0c. API key configuration ─────────────────────────────────────────────
# Use heuristic NLI by default (no model download needed).
# Set FORCE_HEURISTIC_NLI to '0' if you want the full PubMedBERT-BioNLI model.
import os
os.environ.setdefault("FORCE_HEURISTIC_NLI", "1")

import importlib.util
from IPython.display import display, clear_output, HTML

_HAS_WIDGETS = importlib.util.find_spec("ipywidgets") is not None

if _HAS_WIDGETS:
    import ipywidgets as widgets

    # ── Key inputs ──────────────────────────────────────────────────────────
    _or_box   = widgets.Password(placeholder="sk-or-v1-…",             description="OpenRouter:",
                                  layout=widgets.Layout(width="480px"))
    _anth_box = widgets.Password(placeholder="sk-ant-…",               description="Anthropic: ",
                                  layout=widgets.Layout(width="480px"))
    _umls_box = widgets.Password(placeholder="paste UMLS API key here", description="UMLS:      ",
                                  layout=widgets.Layout(width="480px"))

    _btn = widgets.Button(description="Set All Keys", button_style="primary",
                          icon="check", layout=widgets.Layout(width="140px"))
    _out = widgets.Output()

    def _apply(_b):
        with _out:
            clear_output()
            if _or_box.value.strip():
                os.environ["OPENROUTER_API_KEY"] = _or_box.value.strip()
                print(f"  ✓ OpenRouter key set ({len(_or_box.value.strip())} chars)")
            else:
                print("  —  OpenRouter skipped (CoT generation will use local fallback)")

            if _anth_box.value.strip():
                os.environ["ANTHROPIC_API_KEY"] = _anth_box.value.strip()
                print(f"  ✓ Anthropic key set ({len(_anth_box.value.strip())} chars)")
            else:
                print("  —  Anthropic skipped")

            if _umls_box.value.strip():
                os.environ["UMLS_API_KEY"] = _umls_box.value.strip()
                print(f"  ✓ UMLS key set ({len(_umls_box.value.strip())} chars) — density + specificity enabled")
            else:
                print("  —  UMLS skipped (Stage 5 signals will show configured=False)")

    _btn.on_click(_apply)
    display(HTML("<b>API Keys</b> — fill in what you have, leave the rest blank"))
    display(HTML(
        "<small>OpenRouter: <a href='https://openrouter.ai' target='_blank'>openrouter.ai</a>"
        " &nbsp;|&nbsp; UMLS: <a href='https://uts.nlm.nih.gov/uts/signup-login' target='_blank'>"
        "uts.nlm.nih.gov</a></small>"
    ))
    display(widgets.VBox([_or_box, _anth_box, _umls_box, _btn]))
    display(_out)
else:
    # No widgets — prompt for manual entry
    print("ipywidgets not available. Set keys manually before running Stage 1:")
    print('  os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."')
    print('  os.environ["ANTHROPIC_API_KEY"]  = "sk-ant-..."')
    print('  os.environ["UMLS_API_KEY"]        = "your-umls-key"')

In [ ]:
# ── 0d. Import all pipeline modules ────────────────────────────────────────
import warnings, json, time, traceback
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

_ok = {}
for mod, sym in [
    ('utils.cot_generator',          'generate'),
    ('utils.concept_extractor',      'extract_concepts'),
    ('utils.hybrid_checker',         'build_entailment_records'),
    ('utils.guards',                 'derive_guards'),
    ('utils.umls_api_linker',        'is_configured'),
    ('utils.umls_density_scorer',    'score_density'),
    ('utils.umls_specificity_scorer','score_specificity'),
]:
    try:
        m = __import__(mod, fromlist=[sym])
        _ok[mod] = True
        print(f"  ✓ {mod}")
    except Exception as e:
        _ok[mod] = False
        print(f"  ✗ {mod}: {e}")

from utils.cot_generator          import generate as generate_cot
from utils.concept_extractor      import extract_concepts
from utils.hybrid_checker         import build_entailment_records
from utils.guards                 import derive_guards, GuardConfig, lexical_jaccard
from utils.umls_api_linker        import is_configured as umls_configured
from utils.umls_density_scorer    import score_density
from utils.umls_specificity_scorer import score_specificity

GUARD_CFG = GuardConfig()

print()
print(f"  OpenRouter ready : {bool(os.environ.get('OPENROUTER_API_KEY'))}")
print(f"  Anthropic ready  : {bool(os.environ.get('ANTHROPIC_API_KEY'))}")
print(f"  UMLS configured  : {umls_configured()}")
print(f"  Heuristic NLI    : {os.environ.get('FORCE_HEURISTIC_NLI', '1') == '1'}")
print()
print("  ↑ Re-run this cell after entering your OpenRouter key above.")
if not umls_configured():
    print()
    print("  ℹ  UMLS not configured — density and specificity signals will show configured=False.")
    print("     NLI signal and visualisations will still work fully.")

## Stage 1 — CoT Generator

Calls an LLM to produce numbered reasoning steps for a biomedical question.  
If no API key is set, falls back to 5 generic template steps (provider = `local`).

In [ ]:
# ── 1a. Configuration ───────────────────────────────────────────────────────
import os
from IPython.display import display, HTML
import importlib.util

_HAS_WIDGETS = importlib.util.find_spec("ipywidgets") is not None

# Default model — fast and free on OpenRouter. Change via dropdown below.
_MODEL_OPTIONS = {
    "Claude Haiku (fast, recommended)":  "anthropic/claude-haiku-4-5",
    "GPT-4o Mini (OpenAI)":              "openai/gpt-4o-mini",
    "Gemini Flash (Google)":             "google/gemini-flash-1.5",
    "Llama 3.3 70B (free)":              "meta-llama/llama-3.3-70b-instruct:free",
}

TEST_QUESTION = (
    "Does aspirin reduce the risk of myocardial infarction "
    "in patients with cardiovascular disease?"
)

if _HAS_WIDGETS:
    import ipywidgets as widgets

    _model_dropdown = widgets.Dropdown(
        options=list(_MODEL_OPTIONS.keys()),
        value="Claude Haiku (fast, recommended)",
        description="Model:",
        layout=widgets.Layout(width="420px"),
    )
    display(HTML("<b>Model selection</b> — leave as-is if unsure"))
    display(_model_dropdown)
    MODEL_ID = _MODEL_OPTIONS[_model_dropdown.value]

    def _on_model_change(change):
        global MODEL_ID
        MODEL_ID = _MODEL_OPTIONS[change["new"]]
        print(f"  Model set to: {MODEL_ID}")

    _model_dropdown.observe(_on_model_change, names="value")
else:
    MODEL_ID = "anthropic/claude-haiku-4-5"

PREFER = "openrouter"

print(f"Question : {TEST_QUESTION}")
print(f"Model    : {MODEL_ID}")
print(f"Provider : {PREFER}")

In [ ]:
# ── 1b. Run CoT generation ──────────────────────────────────────────────────
t0  = time.time()
cot = generate_cot(TEST_QUESTION, prefer=PREFER, model=STAGE1_MODEL)
elapsed = round(time.time() - t0, 2)

STEPS    = cot['steps']
PROVIDER = cot['provider']
MODEL_ID = cot['model']

print(f"Provider : {PROVIDER}  |  Model : {MODEL_ID}  |  Time : {elapsed}s")
print(f"Steps    : {len(STEPS)}")
print()

for i, step in enumerate(STEPS, 1):
    print(f"  Step {i:2d}: {step}")

if PROVIDER == 'local':
    print()
    print("⚠  provider='local' means all API calls failed.")
    print("   Set OPENROUTER_API_KEY in cell 0c and re-run.")

In [ ]:
# ── 1c. Validate step quality ───────────────────────────────────────────────
checks = {
    'At least 3 steps returned':         len(STEPS) >= 3,
    'All steps non-empty strings':        all(isinstance(s, str) and len(s.strip()) > 0 for s in STEPS),
    'All steps > 15 chars (not trivial)': all(len(s) > 15 for s in STEPS),
    'Real LLM was called (not fallback)': PROVIDER != 'local',
}

all_pass = True
for name, result in checks.items():
    icon = '✓' if result else '✗'
    print(f"  {icon}  {name}")
    if not result: all_pass = False

print()
print("Stage 1:", "PASS ✓" if all_pass else "WARN — check API key")

## Stage 2 — Concept Extractor

Extracts biomedical surface candidates (n-grams, acronyms) from each step  
and links them to UMLS concepts (CUIs) if the UMLS API is configured.

In [ ]:
# ── 2a. Extract concepts ────────────────────────────────────────────────────
t0       = time.time()
CONCEPTS = extract_concepts(STEPS, scispacy_when='never', top_k=5)
elapsed  = round(time.time() - t0, 2)

total_cands = sum(len(c) for c in CONCEPTS)
valid_cands = sum(1 for sc in CONCEPTS for c in sc if c.get('valid'))

print(f"Elapsed         : {elapsed}s")
print(f"Total candidates: {total_cands}  (across {len(STEPS)} steps)")
print(f"Valid (UMLS CUI): {valid_cands}")
print(f"UMLS configured : {umls_configured()}")
if not umls_configured():
    print()
    print("  ℹ  UMLS not configured — concepts will have no CUI.")
    print("     Set UMLS_API_KEY for full concept linking.")

In [ ]:
# ── 2b. Show concepts per step ──────────────────────────────────────────────
rows = []
for i, (step, cands) in enumerate(zip(STEPS, CONCEPTS)):
    for c in cands[:3]:
        rows.append({
            'Step':       i + 1,
            'Step text':  step[:55] + '...' if len(step) > 55 else step,
            'Surface':    c.get('surface', '?'),
            'Name':       c.get('name') or c.get('surface', '?'),
            'CUI':        c.get('cui', '—'),
            'Confidence': round(float((c.get('scores') or {}).get('confidence', 0)), 3),
            'Valid':      c.get('valid', False),
        })

if rows:
    df_concepts = pd.DataFrame(rows)
    display(df_concepts.to_string(index=False))
else:
    print("No concept candidates returned.")
    print("This is expected when UMLS_API_KEY is not set.")

## Stage 3 — Hybrid NLI Entailment Checker

Scores each adjacent step-pair for entailment / neutral / contradiction.

- With `FORCE_HEURISTIC_NLI=1` (default): uses token-overlap heuristic (fast, no download)
- With `FORCE_HEURISTIC_NLI=0`: downloads and runs PubMedBERT-BioNLI-LoRA (~420MB)

In [ ]:
# ── 3a. Run NLI on step pairs ───────────────────────────────────────────────
t0    = time.time()
PAIRS = build_entailment_records(STEPS, CONCEPTS)
elapsed = round(time.time() - t0, 2)

label_counts = Counter(p['final_label'] for p in PAIRS)
print(f"Elapsed        : {elapsed}s")
print(f"Adjacent pairs : {len(PAIRS)}  (= {len(STEPS)} steps - 1)")
print(f"Label counts   : {dict(label_counts)}")
print(f"NLI source     : {(PAIRS[0].get('meta') or {}).get('nli_source', '?') if PAIRS else 'n/a'}")

In [ ]:
# ── 3b. Display pair probabilities ──────────────────────────────────────────
rows = []
for p in PAIRS:
    i, j   = p['step_pair']
    probs  = p.get('probs', {})
    rows.append({
        'Pair':          f'{i}→{j}',
        'Premise':       STEPS[i][:50] + '...' if len(STEPS[i]) > 50 else STEPS[i],
        'Hypothesis':    STEPS[j][:50] + '...' if len(STEPS[j]) > 50 else STEPS[j],
        'P(entail)':     round(probs.get('entailment',    0), 3),
        'P(neutral)':    round(probs.get('neutral',       0), 3),
        'P(contra)':     round(probs.get('contradiction', 0), 3),
        'Final label':   p.get('final_label', '?'),
    })

df_nli = pd.DataFrame(rows)

def colour_label(val):
    colours = {'contradiction':'#ffcccc','entailment':'#ccffcc','neutral':'#e8e8e8'}
    return f"background-color: {colours.get(val, 'white')}"

display(df_nli.style.applymap(colour_label, subset=['Final label'])
               .format({'P(entail)':'{:.3f}','P(neutral)':'{:.3f}','P(contra)':'{:.3f}'}))

In [ ]:
# ── 3c. Validate NLI output ─────────────────────────────────────────────────
valid_labels = {'entailment', 'neutral', 'contradiction'}
checks = {
    f'Returns {len(STEPS)-1} pairs (N-1)':  len(PAIRS) == len(STEPS) - 1,
    'All probs sum to ≈ 1.0':
        all(abs(sum(p['probs'].values()) - 1.0) < 0.05 for p in PAIRS),
    'All final_labels are valid':
        all(p.get('final_label') in valid_labels for p in PAIRS),
}
for name, ok in checks.items():
    print(f"  {'✓' if ok else '✗'}  {name}")
print()
print("Stage 3:", "PASS ✓" if all(checks.values()) else "FAIL ✗")

## Stage 4 — Guard Signal Derivation

Guard signals are qualitative tags computed **on top of** NLI probabilities.

| Guard | Fires when |
|---|---|
| `lexical_duplicate` | Adjacent steps are ≥ 90% lexically identical (wasted reasoning) |
| `caution_band` | Top two label probabilities are very close (the model is uncertain) |
| `direction_conflict` | NLI is asymmetric: A→B entails but B→A contradicts |

In [ ]:
# ── 4a. Compute guard signals for each pair ─────────────────────────────────
def _reverse_probs(steps, concepts, i, j):
    try:
        rev = build_entailment_records([steps[j], steps[i]],
                                       [concepts[j] if j < len(concepts) else [],
                                        concepts[i] if i < len(concepts) else []])
        return rev[0]['probs'] if rev else None
    except Exception:
        return None

GUARDED_PAIRS = []
for p in PAIRS:
    i, j      = p['step_pair']
    rev_probs = _reverse_probs(STEPS, CONCEPTS, i, j)
    guards    = derive_guards(
        premise       = STEPS[i] if i < len(STEPS) else '',
        hypothesis    = STEPS[j] if j < len(STEPS) else '',
        probs         = p['probs'],
        reverse_probs = rev_probs,
        config        = GUARD_CFG,
    )
    GUARDED_PAIRS.append({**p, 'guards': guards, 'reverse_probs': rev_probs})

all_guards = [g for p in GUARDED_PAIRS for g in p['guards']]
print(f"Total guards fired : {len(all_guards)}")
print(f"Guard breakdown    : {dict(Counter(all_guards)) or 'none'}")

In [ ]:
# ── 4b. Display guard signals per pair ──────────────────────────────────────
rows = []
for p in GUARDED_PAIRS:
    i, j   = p['step_pair']
    probs  = p['probs']
    rprobs = p.get('reverse_probs') or {}
    rows.append({
        'Pair':           f'{i}→{j}',
        'Label':          p['final_label'],
        'P(contra) fwd':  round(probs.get('contradiction', 0), 3),
        'P(entail) rev':  round(rprobs.get('entailment', 0), 3) if rprobs else '—',
        'Guards':         ', '.join(p['guards']) or 'none',
    })

df_guards = pd.DataFrame(rows)
display(df_guards.to_string(index=False))

## Stage 5 — UMLS Signal Computation

Runs the two new UMLS chain-level scorers on the concepts extracted in Stage 2.

| Signal | What it measures | Output |
|---|---|---|
| **Density** | Valid UMLS concept count per word, per step | `overall_risk` (0–1), slope, leakage onset step |
| **Specificity** | Average hierarchy depth of concepts per step | `overall_specificity_score` (0–1), depth slope, abstraction leaps |

Both signals require `UMLS_API_KEY` to be configured. If not set, they return `configured=False` and the visualisations in Stage 6 will show NLI only.

In [ ]:
# ── 5a. Run density + specificity scorers ───────────────────────────────────
t0          = time.time()
DENSITY     = score_density(CONCEPTS, steps=STEPS)
SPECIFICITY = score_specificity(CONCEPTS)
elapsed     = round(time.time() - t0, 2)

print(f"Elapsed                    : {elapsed}s")
print()
print(f"── Density ──")
print(f"  configured               : {DENSITY['configured']}")
print(f"  overall_risk             : {DENSITY['overall_risk']}")
print(f"  slope                    : {DENSITY['slope']}")
print(f"  leakage_onset_step       : {DENSITY['leakage_onset_step']}")
print()
print(f"── Specificity ──")
print(f"  configured               : {SPECIFICITY['configured']}")
print(f"  overall_specificity_score: {SPECIFICITY['overall_specificity_score']}")
print(f"  depth_slope              : {SPECIFICITY['depth_slope']}")
print(f"  abstraction_leaps        : {SPECIFICITY['abstraction_leaps']}")

if not DENSITY['configured']:
    print()
    print("  ℹ  UMLS not configured — set UMLS_API_KEY for density + specificity signals.")
    print("     Stage 6 visualisations will show NLI signal only.")

In [ ]:
# ── 5b. Per-step breakdown table ────────────────────────────────────────────
rows = []
for i, step in enumerate(STEPS):
    d_row = next((r for r in DENSITY['per_step']     if r['step_index'] == i), {})
    s_row = next((r for r in SPECIFICITY['per_step'] if r['step_index'] == i), {})
    rows.append({
        'Step':           i,
        'Step text':      step[:60] + '...' if len(step) > 60 else step,
        'Valid concepts': d_row.get('valid_concept_count', '—'),
        'Word count':     d_row.get('word_count', '—'),
        'Density':        round(d_row['density'], 4) if 'density' in d_row else '—',
        'Avg depth':      round(s_row['avg_depth'], 2) if s_row.get('avg_depth') is not None else '—',
        'Depth concepts': s_row.get('concept_count', '—'),
    })

df_umls = pd.DataFrame(rows)
display(df_umls.to_string(index=False))

if not DENSITY['configured']:
    print()
    print("  ℹ  Density and depth columns show '—' because UMLS is not configured.")

## Stage 6 — Signal Alignment & Visualisation

Four plots that combine all three signals.

| Plot | What it shows |
|---|---|
| **6a** Timeline | All three signals per step — where they rise and fall together |
| **6b** Heatmap | Step × signal risk grid — which steps are flagged by which signals |
| **6c** Chain grades | One bar per signal — the overall verdict on the chain |
| **6d** NLI contradiction matrix | P(contradiction) for every step pair |

In [ ]:
# ── 6a. Per-step signal timeline ────────────────────────────────────────────
#
# NLI is pairwise (N-1 scores). We assign each pair score to its destination
# step j:  nli_risk[j] = 1 - P(entailment) for pair (j-1 → j).
# Step 0 has no incoming pair so it gets None.

n = len(STEPS)

nli_risk = [None] + [
    1.0 - p['probs'].get('entailment', 0.0)
    for p in PAIRS
]

density_vals = [
    next((r['density'] for r in DENSITY['per_step'] if r['step_index'] == i), None)
    for i in range(n)
]

depth_vals = [
    next((r['avg_depth'] for r in SPECIFICITY['per_step'] if r['step_index'] == i), None)
    for i in range(n)
]

def _norm(values):
    """Min-max normalise a list that may contain None."""
    clean = [v for v in values if v is not None]
    if not clean or max(clean) == min(clean):
        return values
    lo, hi = min(clean), max(clean)
    return [None if v is None else (v - lo) / (hi - lo) for v in values]

nli_norm     = _norm(nli_risk)
density_norm = _norm(density_vals)
depth_norm   = _norm(depth_vals)

xs = list(range(n))

fig, ax = plt.subplots(figsize=(max(8, n * 1.2), 4))

# NLI risk line (always present)
nli_xs = [x for x, v in zip(xs, nli_norm) if v is not None]
nli_ys = [v for v in nli_norm if v is not None]
ax.plot(nli_xs, nli_ys, 'o-', color='#C44E52', linewidth=2, label='NLI risk (1 − P(entail))')

# Density line (UMLS-dependent)
if DENSITY['configured']:
    d_xs = [x for x, v in zip(xs, density_norm) if v is not None]
    d_ys = [v for v in density_norm if v is not None]
    ax.plot(d_xs, d_ys, 's--', color='#4C72B0', linewidth=2, label='UMLS density (normalised)')

# Depth line (UMLS-dependent)
if SPECIFICITY['configured']:
    sp_xs = [x for x, v in zip(xs, depth_norm) if v is not None]
    sp_ys = [v for v in depth_norm if v is not None]
    ax.plot(sp_xs, sp_ys, '^:', color='#55A868', linewidth=2, label='UMLS depth (normalised)')

# Leakage onset marker
onset = DENSITY.get('leakage_onset_step')
if onset is not None:
    ax.axvline(onset, color='red', linestyle='--', alpha=0.6, linewidth=1.5,
               label=f'Leakage onset (step {onset})')

# Abstraction leap shading
for leap in SPECIFICITY.get('abstraction_leaps', []):
    ax.axvspan(leap['from_step'] - 0.3, leap['to_step'] + 0.3,
               alpha=0.12, color='orange',
               label=f"Abstraction leap ({leap['from_step']}→{leap['to_step']})")

ax.set_xticks(xs)
ax.set_xticklabels([f'Step {i}' for i in xs], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Risk (0 = low, 1 = high)', fontsize=9)
ax.set_title('Per-step Signal Timeline\n(all signals normalised 0–1 for comparison)', fontsize=11)
ax.set_ylim(-0.05, 1.15)
ax.legend(fontsize=8, loc='upper left')
ax.grid(axis='y', alpha=0.3)

if not DENSITY['configured']:
    ax.text(0.99, 0.97, 'UMLS not configured — density + depth lines absent',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=7, color='grey', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6b. Signal agreement heatmap ────────────────────────────────────────────
#
# Rows = steps,  Columns = [NLI risk, Density risk, Specificity risk]
# All values normalised 0–1.  Red = high risk, green = low risk.
# NLI risk at step 0 = 0 (no incoming pair).

nli_col     = [0.0 if v is None else v for v in nli_norm]
density_col = [0.0 if v is None else v for v in density_norm]
depth_col   = [0.0 if v is None else v for v in depth_norm]

# Invert density and depth: high density/depth = low risk
# (density drops = risk rises; depth drops = risk rises)
density_risk_col = [1.0 - v for v in density_col] if DENSITY['configured'] \
                   else [0.0] * n
depth_risk_col   = [1.0 - v for v in depth_col]   if SPECIFICITY['configured'] \
                   else [0.0] * n

hm_data = np.array([nli_col, density_risk_col, depth_risk_col]).T  # shape (n_steps, 3)
col_labels = ['NLI risk', 'Density risk', 'Specificity risk']
row_labels = [f'Step {i}' for i in range(n)]

fig, ax = plt.subplots(figsize=(5, max(3, n * 0.55)))
sns.heatmap(
    hm_data,
    ax=ax,
    xticklabels=col_labels,
    yticklabels=row_labels,
    cmap='RdYlGn_r',
    vmin=0, vmax=1,
    annot=True, fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'Risk (0=low, 1=high)', 'shrink': 0.8},
)
ax.set_title('Signal Agreement Heatmap\n(each cell = risk score per step per signal)', fontsize=10)
ax.set_ylabel('Reasoning step', fontsize=9)

if not DENSITY['configured']:
    ax.text(0.99, -0.08, 'Density + Specificity columns are zero (UMLS not configured)',
            transform=ax.transAxes, ha='right', fontsize=7, color='grey', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6c. Chain grade summary bars ────────────────────────────────────────────
#
# One horizontal bar per signal.  The "report card" for the whole chain.
# NLI chain score = 1 - average P(entailment) across all pairs.

avg_entailment = (
    sum(p['probs'].get('entailment', 0.0) for p in PAIRS) / len(PAIRS)
    if PAIRS else 0.0
)
nli_chain_risk   = round(1.0 - avg_entailment, 3)
density_grade    = DENSITY.get('overall_risk', 0.0)
specificity_grade = SPECIFICITY.get('overall_specificity_score', 0.0)

signals = ['NLI chain risk', 'UMLS density risk', 'UMLS specificity risk']
scores  = [nli_chain_risk, density_grade, specificity_grade]

def _bar_color(v):
    if v < 0.3:   return '#2ca02c'   # green
    if v < 0.6:   return '#ff7f0e'   # orange
    return '#d62728'                  # red

colors = [_bar_color(v) for v in scores]

fig, ax = plt.subplots(figsize=(7, 2.8))
bars = ax.barh(signals, scores, color=colors, height=0.45, edgecolor='white')

for bar, v in zip(bars, scores):
    ax.text(min(v + 0.02, 0.97), bar.get_y() + bar.get_height() / 2,
            f'{v:.3f}', va='center', fontsize=10, fontweight='bold')

ax.axvline(0.3, color='grey', linestyle=':', linewidth=1, alpha=0.7)
ax.axvline(0.6, color='grey', linestyle=':', linewidth=1, alpha=0.7)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Risk score (0 = low, 1 = high)', fontsize=9)
ax.set_title('Chain Grade Summary\n(green < 0.3 | orange 0.3–0.6 | red > 0.6)', fontsize=10)
ax.grid(axis='x', alpha=0.3)

if not DENSITY['configured']:
    ax.text(0.99, 0.04, 'UMLS bars are 0 — UMLS not configured',
            transform=ax.transAxes, ha='right', fontsize=7, color='grey', style='italic')

plt.tight_layout()
plt.show()

print(f"  NLI chain risk            : {nli_chain_risk}")
print(f"  UMLS density risk         : {density_grade}")
print(f"  UMLS specificity risk     : {specificity_grade}")

In [ ]:
# ── 6d. NLI P(contradiction) matrix ─────────────────────────────────────────
#
# N×N matrix where cell [i,j] = P(contradiction) for step pair i→j.
# Only adjacent pairs are scored; non-adjacent cells are grey (NaN).

mat = np.full((n, n), np.nan)
for p in PAIRS:
    i, j = p['step_pair']
    mat[i, j] = p['probs'].get('contradiction', 0)

fig, ax = plt.subplots(figsize=(max(5, n * 0.9), max(4, n * 0.8)))
im = ax.imshow(mat, vmin=0, vmax=1, cmap='RdYlGn_r', aspect='auto')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([f'Step {i}' for i in range(n)], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([f'Step {i}' for i in range(n)], fontsize=8)
ax.set_xlabel('Hypothesis (step j)', fontsize=9)
ax.set_ylabel('Premise (step i)', fontsize=9)

for p in PAIRS:
    i, j = p['step_pair']
    val  = mat[i, j]
    lbl  = p['final_label'][0].upper()
    ax.text(j, i, f"{lbl}\n{val:.2f}",
            ha='center', va='center', fontsize=8,
            color='white' if val > 0.6 else 'black')

plt.colorbar(im, ax=ax, label='P(contradiction)', shrink=0.8)
ax.set_title('NLI P(contradiction) per Step-Pair\n(E=entailment, N=neutral, C=contradiction)', fontsize=10)
plt.tight_layout()
plt.show()